In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass


EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXXXX" # Add your namespace-prefix here

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
   VlmAbilityCreate(
       name=f"{NAMESPACE_PREFIX}.structured-OCR.patient-intake:latest",
       description="Extract relevant information from a patient form",
       worker_release="qwen3-instruct",
       text_prompt="""
          You are given an image of a patient form.
          Your task is to extract structured data that is clearly visible in the image.


           Return ONLY valid JSON.
           Do not include explanation.
           Do not include markdown.
           Do not include commentary.


           ---------------------------------------
           INSTRUCTIONS:


          1. Extract only text that is clearly readable.
          2. Do NOT guess or infer missing values.
          3. If a field is not visible or unreadable, return null.
          4. Preserve capitalization exactly as shown.
          5. Preserve spacing inside fields where applicable.
          6. Do NOT reformat dates unless the format is explicitly clear.
          7. Do NOT fabricate data.
          8. If the image is not a patient form, return:
          {
            "document_type": "unknown"
          }
          9. "surgeries" should be an array of objects, each with "procedure" and "year".
          "allergies" should be an array of objects, each with "allergen" and "reaction".
          "medications" should be an array of objects, each with "name", "dosage", and
          "frequency". Return an empty array for any of these if none are present on the
          form.


                    ---------------------------------------
                    RETURN THIS EXACT JSON STRUCTURE:


          {
            "patient": {
              "first_name": null,
              "last_name": null,
              "date_of_birth": null,
              "age": null,
              "sex": null,
              "phone": null,
              "email": null,
              "address": {
                "street": null,
                "city": null,
                "state": null,
                "zip": null
              }
            },
            "emergency_contact": {
              "name": null,
              "relationship": null,
              "phone": null
            },
            "insurance": {
              "provider": null,
              "member_id": null,
              "group_number": null,
              "policy_holder": null,
              "relationship_to_patient": null
            },
            "visit": {
              "visit_date": null,
              "reason_for_visit": null,
              "symptoms": [],
              "symptom_start_date": null,
              "pain_level_0_to_10": null
            },
            "medical_history": {
              "conditions": [],
              "surgeries": [],
              "allergies": []
            },
            "medications": [],
            "lifestyle": {
              "smoker": null,
              "alcohol_use": null,
              "exercise_frequency": null
            },
            "consent": {
              "hipaa_acknowledged": null,
              "treatment_consent_signed": null,
              "signature_present": null,
              "signature_name": null,
              "signed_date": null
            },
            "extraction_metadata": {
              "source_type": null,
              "confidence": null,
              "fields_requiring_review": []
            }
          }


           ---------------------------------------


        If multiple values appear for a field, return the most prominent official value.
        Return only the JSON object.


""",
       transform_into=TransformInto(),
       config=InferRuntimeConfig(
           max_new_tokens=1000,
           image_size=1000
       ),
       is_public=False
   )
]



In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image

In [ ]:
from pathlib import Path


pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.structured-OCR.patient-intake:latest"
   )
])


with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path("/content/sample_img.jpg") # change to the path of your sample image
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
     print(json.dumps(result, indent=2))

print("Done")
